# Goldfish Model - Direct EdgeTPU Export
## Export Trained Model Directly to EdgeTPU Format

This notebook exports your trained `goldfish_best.pt` model directly to EdgeTPU format with proper INT8 quantization.

**Why this approach:**
- YOLOv8's `edgetpu` export format handles full INT8 quantization correctly
- Previous TFLite exports created FLOAT32 models (not compatible with EdgeTPU acceleration)
- This creates a properly quantized model that runs at 20-30 FPS on Coral TPU

**Instructions:**
1. Upload your `goldfish_best.pt` file
2. Run all cells
3. Download the `goldfish_best_full_integer_quant_edgetpu.tflite` file

**Time:** ~5-10 minutes

## Step 1: Install YOLOv8

In [ ]:
print("📦 Installing YOLOv8...")
!pip install ultralytics -q

from ultralytics import YOLO
import ultralytics
print(f"✅ Ultralytics version: {ultralytics.__version__}")

## Step 2: Upload Your Trained Model

Click the upload button below and select `goldfish_best.pt` from your computer.

In [ ]:
from google.colab import files
import os

print("📤 Upload your goldfish_best.pt file:")
print("(Click 'Choose Files' below)\n")

uploaded = files.upload()

if 'goldfish_best.pt' in uploaded:
    size_mb = len(uploaded['goldfish_best.pt']) / (1024*1024)
    print(f"\n✅ Uploaded goldfish_best.pt ({size_mb:.1f} MB)")
else:
    print("\n❌ Please upload goldfish_best.pt")
    print(f"You uploaded: {list(uploaded.keys())}")

## Step 3: Install EdgeTPU Compiler

This is needed for the EdgeTPU export format.

In [ ]:
print("📦 Installing EdgeTPU compiler...\n")

!curl https://packages.cloud.google.com/apt/doc/apt-key.gpg | sudo apt-key add -
!echo "deb https://packages.cloud.google.com/apt coral-edgetpu-stable main" | sudo tee /etc/apt/sources.list.d/coral-edgetpu.list
!sudo apt-get update -qq
!sudo apt-get install edgetpu-compiler -y -qq

print("\n✅ EdgeTPU compiler installed!")
!edgetpu_compiler --version

## Step 4: Export to EdgeTPU Format

This exports directly to EdgeTPU format with full INT8 quantization (input/output + weights).

**Note:** This will download calibration data automatically.

In [ ]:
import time

print("="*70)
print("EXPORTING TO EDGETPU FORMAT (FULL INT8 QUANTIZATION)")
print("="*70)

print("\n📦 Loading goldfish_best.pt...")
model = YOLO('goldfish_best.pt')
print("✅ Model loaded!")

print("\n🔄 Exporting to EdgeTPU format...")
print("This will take 5-10 minutes...")
print("(Installing dependencies + quantization + compilation)\n")

start = time.time()

# Export to EdgeTPU with full integer quantization
try:
    model.export(
        format='edgetpu',
        imgsz=640,
    )
    elapsed = (time.time() - start) / 60
    print(f"\n✅ Export complete in {elapsed:.1f} minutes!")
except Exception as e:
    print(f"\n❌ Export failed: {e}")
    print("\nThis may be due to missing dependencies or Python version.")
    print("Colab uses Python 3.10+ which should work.")

## Step 5: Find and Download EdgeTPU Model

In [ ]:
import glob
import shutil

print("🔍 Locating EdgeTPU model...\n")

# Find the EdgeTPU file
edgetpu_files = glob.glob('*_saved_model/*_full_integer_quant_edgetpu.tflite')
if not edgetpu_files:
    edgetpu_files = glob.glob('*_saved_model/*_edgetpu.tflite')
if not edgetpu_files:
    edgetpu_files = glob.glob('*edgetpu.tflite')

if edgetpu_files:
    edgetpu_file = edgetpu_files[0]
    print(f"✅ Found: {edgetpu_file}")
    
    # Copy to a simple filename
    output_name = 'goldfish_best_edgetpu.tflite'
    shutil.copy(edgetpu_file, output_name)
    
    size_mb = os.path.getsize(output_name) / (1024*1024)
    print(f"Size: {size_mb:.2f} MB")
    
    # Check if it's properly quantized
    if 'full_integer_quant' in edgetpu_file:
        print("✅ Model has full INT8 quantization!")
    
    print(f"\n📥 Downloading {output_name}...")
    files.download(output_name)
    
    print("\n✅ Download complete!")
    print("\nNext steps:")
    print("1. The file is now in your Downloads folder")
    print("2. Upload it to your Raspberry Pi")
    print("3. Test goldfish detection with the EdgeTPU!")
    print("\nExpected performance: 20-30 FPS on Coral TPU")
else:
    print("❌ EdgeTPU model not found!")
    print("\nSearching for any EdgeTPU-related files:")
    !find . -name "*edgetpu*" -o -name "*saved_model*" | head -20
    print("\nPlease share this output so we can troubleshoot.")

## ✅ Complete!

You should now have `goldfish_best_edgetpu.tflite` in your Downloads folder.

**File Info:**
- Format: TFLite with full INT8 quantization for Edge TPU
- Size: ~3-4 MB
- Ready to run on Coral USB Accelerator
- Expected speed: 20-30 FPS

**Key Difference from Previous Export:**
- This model has INT8 inputs/outputs (not FLOAT32)
- EdgeTPU can accelerate this model properly
- Previous models ran on CPU (~1 FPS)
- This model will run on TPU (~25 FPS)

**Next:** Upload this file to your Raspberry Pi and test!